Пробвай да промениш дължината на клиповете, така че да визмаш предвид само първите няколко минути или първите 50%.
Пробвай да промениш сийда. Може тоя да е скапан.
#Пробвай да направиш системата да работи за един определен човек. Не за всичо и висчки. 

In [1]:
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import copy
from pathlib import Path
import torch.profiler
from enum import Enum
import optuna
from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
import os
import random
from sklearn.model_selection import StratifiedGroupKFold
import polars as pl

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [2]:
def seed_all(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    
seed = 42
seed_all(seed)

In [3]:
class ModelType(Enum):
    CNN = 1
    LSTM = 2
    CNN_LSTM = 3
    LSTM_CNN = 4
    CNN_LSTM_Fusion = 5
    CNN_Transformer = 6
    MLP = 7

class EvalMetric(Enum):
    accuracy = 1
    F1 = 2
    custom = 3

class ExperimentType(Enum):
    fixed_model_params = 1
    best_model = 2

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [5]:
root = "../../../"
data_path = f'{root}data/'
training_data_path = data_path + "Final Training Data/"

In [6]:
class CNN(nn.Module):
    def __init__(self, input_channels, output_channels, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1*output_c_multip
        C3 = C2*output_c_multip

        self.features = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, 5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.Conv1d(C2, C3, 3, padding=1),
            nn.BatchNorm1d(C3),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(C3, C3),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(C3, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)



        
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()

        H = hidden_size  

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=num_layers,   
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return self.head(x)

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        weights = self.attn(x)             
        weights = torch.softmax(weights, dim=1)

        pooled = torch.sum(x * weights, dim=1)
        return pooled





class CNN_LSTM(nn.Module):
    def __init__(self, input_channels, output_channels, hidden_size, num_layers, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1 * output_c_multip
        C3 = C2 * output_c_multip
        H = hidden_size  

        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, kernel_size=5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Conv1d(C1, C2, kernel_size=5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Conv1d(C2, C3, kernel_size=3, padding=1),
            nn.BatchNorm1d(C3),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=C3,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.norm = nn.LayerNorm(H)

        self.pool = AttentionPooling(H)

        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H, H // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H // 2, num_classes)
        )

    def forward(self, x):
        # x: (B, T, F)
        x = x.permute(0, 2, 1)  
        x = self.cnn(x)         
        x = x.permute(0, 2, 1)  
        
        x, _ = self.lstm(x)      
        x = self.norm(x)
        x = self.pool(x)         

        return self.head(x)




class CNN_Transformer(nn.Module):
    def __init__(self, input_channels, output_channels, num_layers, num_classes, feature_dim, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1*output_c_multip
        C3 = C2*output_c_multip

        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, 5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.Conv1d(C2, C3, 5, padding=2),
            nn.BatchNorm1d(C3),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=C3,
                nhead=4,
                batch_first=True
            ),
            num_layers=num_layers
        )

        self.fusion = nn.Sequential(
            nn.Linear(C3 + feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, x_engineered):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        x = self.transformer(x)
        x = x.mean(dim=1)

        x = torch.cat([x, x_engineered], dim=1)
        return self.fusion(x)



        
class LSTM_CNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, activation_fn, dropout):
        super().__init__()

        H = hidden_size

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True
        )

        self.cnn = nn.Sequential(
            nn.Conv1d(H, H, 5, padding=2),
            nn.BatchNorm1d(H),
            activation_fn(),

            nn.MaxPool1d(2),

            nn.Conv1d(H, H, 3, padding=1),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(H, H),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)





class CNN_LSTM_Fusion(nn.Module):
    def __init__(self, input_channels, output_channels, hidden_size, num_layers, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1 * output_c_multip
        H = hidden_size  

        # CNN branch
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, kernel_size=5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.MaxPool1d(2),
        )

        # LSTM branch
        self.lstm = nn.LSTM(
            input_size=input_channels,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True
        )

        # fusion head
        self.classifier = nn.Sequential(
            nn.Linear(C2 + H, 96),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(96, num_classes)
        )

    def forward(self, x):
        # CNN
        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.cnn(x_cnn)
        x_cnn = x_cnn.mean(dim=-1)

        # LSTM
        x_lstm, _ = self.lstm(x)
        x_lstm = x_lstm.mean(dim=1)

        x = torch.cat([x_cnn, x_lstm], dim=1)
        return self.classifier(x)



        
class MLP(nn.Module):
    def __init__(self, input_features, output_channles, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channles
        C2 = C1*output_c_multip

        self.net = nn.Sequential(
            nn.Linear(input_features, C1),
            nn.LayerNorm(C1),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(C1, C2),
            nn.LayerNorm(C2),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(C2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
def load_npz(path, subject_root_path, subject):
    path = Path(path)
    data = np.load(f"{path}.npz", allow_pickle=True)
    relative_windows_times = np.load(subject_root_path / f"{subject}_window_times.npz", allow_pickle=True)
    return data["X"], data["y"], relative_windows_times["window_times"]

def load_subject(cache_dir, subject, use_raw, use_engineered):
    X_raw = None
    X_engineered = None
    y = None

    cache_dir = Path(cache_dir)
    subject_root_path = cache_dir / subject
    if use_raw:
        X_raw, y, relative_windows_times = load_npz(subject_root_path / f"{subject}_raw", subject_root_path, subject)

    if use_engineered:
        X_engineered, y_engineered, relative_windows_times = load_npz(subject_root_path / f"{subject}_extracted", subject_root_path, subject)

        if y is None:
            y = y_engineered
        elif not np.array_equal(y, y_engineered):
            raise ValueError("Labels do not match")

    return X_raw, X_engineered, y, relative_windows_times

def get_subjects(cache_dir, use_raw, use_engineered):
    cache_dir = Path(cache_dir)
    subjects = set()

    if use_raw:
        subjects.update(
            p.stem.removesuffix("_raw")
            for p in Path(cache_dir).rglob("*_raw.npz")
        )

    if use_engineered:
        subjects.update(
            p.stem.removesuffix("_extracted")
            for p in Path(cache_dir).rglob("*_extracted.npz")
        )

    return sorted(subjects)

In [8]:
def compute_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def compute_f1_score(y_true, y_pred):
    return f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
    )

In [9]:
def get_loader(settings, X_train_raw, X_train_extracted, y_train):
    sampler = None
    shuffle = True
    batch_size = settings["batch size"]
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = class_counts.sum() / class_counts
        
        weights = weights / weights.mean()
        #weights = 1.0 / class_counts
        
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])

    if X_train_raw is None:
        X_train_raw = torch.empty((len(y_train), 0, 0), dtype=torch.float32)

    if X_train_extracted is None:
        X_train_extracted = torch.empty((len(y_train), 0), dtype=torch.float32)

    return DataLoader(
                    TensorDataset(X_train_raw, X_train_extracted, y_train),
                    batch_size=batch_size,
                    shuffle=shuffle,
                    sampler=sampler,
                    #num_workers=1,
                    pin_memory=True,
                    #prefetch_factor=2
                ), loss_fn

In [10]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch_raw, X_batch_extracted, y_batch) in enumerate(loader):
        X_batch_raw = X_batch_raw.to(device, non_blocking=True)
        X_batch_extracted = X_batch_extracted.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model_forward(model, X_batch_raw, X_batch_extracted)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

def validate_epoch(model, X_val_raw, X_val_extracted, y_val, metadata):

    model.eval()

    with torch.no_grad():

        X_val_raw = X_val_raw.to(device)
        X_val_extracted = X_val_extracted.to(device)
        y_val = y_val.to(device)

        logits = model_forward(
            model,
            X_val_raw,
            X_val_extracted
        )

        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)
        preds = preds.cpu().numpy()

        if metadata["eval_metric"] == EvalMetric.accuracy:
            val_metric = compute_accuracy(
                y_val.cpu(),
                preds
            )

        elif metadata["eval_metric"] == EvalMetric.F1:
            val_metric = compute_f1_score(
                y_val.cpu(),
                preds
            )
        else:
            confidences, preds = torch.max(probs, dim=1)

            mask = confidences >= 0.8

            if mask.sum() == 0:
                val_metric = 0
            else:
                coverage = mask.float().mean().item()
                precision = (
                    preds[mask] == y_val[mask]
                ).float().mean().item()

                val_metric = precision * coverage

    return val_metric

def model_forward(model, x_raw, x_extracted):

    has_raw = x_raw.numel() > 0
    has_extracted = x_extracted.numel() > 0

    if has_raw and has_extracted:
        return model(x_raw, x_extracted)

    if has_raw:
        return model(x_raw)

    if has_extracted:
        return model(x_extracted)

    raise ValueError("No input features provided")

In [11]:
def get_model(model_settings, X_train_main_raw, X_train_main_extracted, y_train_main):
    match model_settings["activation_fn"]:
        case "relu":
            activation_fn = nn.ReLU
        case "gelu":
            activation_fn = nn.GELU

    dropout = model_settings["dropout"]
    hidden_size = model_settings["hidden_size"]
    num_layers = model_settings["num_layers"]
    output_channles = model_settings["output_channels"]
    output_c_multip = model_settings["output_c_multip"]
    
    input_channels_raw = X_train_main_raw.shape[-1]
    input_channels_extracted = X_train_main_extracted.shape[1]
    num_classes = len(torch.unique(y_train_main))
    
    match model_settings["model"]:
        case ModelType.CNN:
            model = CNN(
                input_channels_raw,
                output_channles,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.LSTM:
            model = LSTM(
                input_channels_raw,
                hidden_size,
                num_layers,
                num_classes,
                dropout
            ).to(device)
        case ModelType.CNN_LSTM:
            model = CNN_LSTM(
                input_channels_raw,
                output_channles,
                hidden_size, 
                num_layers,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.LSTM_CNN:
            model = LSTM_CNN(
                input_channels_raw,
                hidden_size, 
                num_layers, 
                num_classes, 
                activation_fn, 
                dropout
            ).to(device)
        case ModelType.CNN_Transformer:
            model = CNN_Transformer(
                input_channels_raw,
                output_channles,
                num_layers,
                num_classes,
                input_channels_extracted,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.CNN_LSTM_Fusion:
            model = CNN_LSTM_Fusion(
                input_channels_raw,
                output_channles,
                hidden_size,
                num_layers,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.MLP:
            model = MLP(
                input_channels_extracted,
                output_channles,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)

    return model

In [12]:
def create_subject_split(relative_times, n_blocks = 20):

    blocks = np.floor(relative_times * n_blocks).astype(int)

    return blocks

In [13]:
def train(
    X_raw, X_extracted, y, 
    relative_windows_times,
    subject_idx, test_subject,
    settings, metadata,
):
    # ------------------------------------------------------------------
    # Convert to CPU tensors
    # ------------------------------------------------------------------
    if X_raw is not None:
        X_raw = torch.tensor(X_raw, dtype=torch.float32)

    if X_extracted is not None:
        X_extracted = torch.tensor(X_extracted, dtype=torch.float32)

    y = torch.tensor(y, dtype=torch.long)

    # ------------------------------------------------------------------
    # Split train / validation
    # ------------------------------------------------------------------
    split_source = X_raw if X_raw is not None else X_extracted
    
    split_np = split_source.numpy()
    y_np = y.numpy()
    
    groups = create_subject_split(
        relative_windows_times
    )
    
    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )
    
    splits = list(
        splitter.split(
            split_np,
            y_np,
            groups
        )
    )
    
    test_idx = splits[0][1]
    val_idx = splits[1][1]
    
    train_idx = np.setdiff1d(
        np.arange(len(y)),
        np.concatenate([
            test_idx,
            val_idx
        ])
    )

    assert len(set(groups[train_idx]) & set(groups[val_idx])) == 0
    assert len(set(groups[train_idx]) & set(groups[test_idx])) == 0
    assert len(set(groups[val_idx]) & set(groups[test_idx])) == 0
    # ------------------------------------------------------------------
    # Slice data
    # ------------------------------------------------------------------
    X_train_raw = None
    X_val_raw = None
    X_test_raw = None

    X_train_extracted = None
    X_val_extracted = None
    X_test_extracted = None
    
    if X_raw is not None:
        X_train_raw = X_raw[train_idx]
        X_val_raw = X_raw[val_idx]
        X_test_raw = X_raw[test_idx]

        mean = X_train_raw.mean(dim=(0,1), keepdim=True)
        std = X_train_raw.std(dim=(0,1), keepdim=True)
    
        X_train_raw = (X_train_raw - mean) / (std + 1e-8)
        X_val_raw = (X_val_raw - mean) / (std + 1e-8)
        X_test_raw = (X_test_raw - mean) / (std + 1e-8)

    if X_extracted is not None:
        X_train_extracted = X_extracted[train_idx]
        X_val_extracted = X_extracted[val_idx]
        X_test_extracted = X_extracted[test_idx]

        mean = X_train_extracted.mean(dim=(0,1), keepdim=True)
        std = X_train_extracted.std(dim=(0,1), keepdim=True)
    
        X_train_extracted = (X_train_extracted - mean) / (std + 1e-8)
        X_val_extracted = (X_val_extracted - mean) / (std + 1e-8)
        X_test_extracted = (X_test_extracted - mean) / (std + 1e-8)
        
    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    

    
    if metadata["eval_metric"] == EvalMetric.custom:
        y_val = y_val.float()


    if X_train_raw is None:
        X_train_raw = torch.empty(
            (len(y_train), 0, 0),
            dtype=torch.float32
        )
    
    if X_val_raw is None:
        X_val_raw = torch.empty(
            (len(y_val), 0, 0),
            dtype=torch.float32
        )

    if X_test_raw is None:
        X_test_raw = torch.empty(
            (len(y_test), 0, 0),
            dtype=torch.float32
        )
    
    if X_train_extracted is None:
        X_train_extracted = torch.empty(
            (len(y_train), 0),
            dtype=torch.float32
        )
    
    if X_val_extracted is None:
        X_val_extracted = torch.empty(
            (len(y_val), 0),
            dtype=torch.float32
        )

    if X_test_extracted is None:
        X_test_extracted = torch.empty(
            (len(y_test), 0),
            dtype=torch.float32
        )

    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    model = get_model(
        settings["model settings"],
        X_train_raw,
        X_train_extracted,
        y_train,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=settings["training settings"]["learning rate"],
        weight_decay=settings["training settings"]["weight decay"],
    )

    loader, loss_fn = get_loader(
        settings["training settings"],
        X_train_raw,
        X_train_extracted,
        y_train,
    )

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    best_val = -1
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(settings["training settings"]["epoch"]):

        train_epoch(
            model,
            loader,
            optimizer,
            loss_fn,
            settings["training settings"]["accumulation_steps"],
        )

        val_score = validate_epoch(
            model,
            X_val_raw,
            X_val_extracted,
            y_val,
            metadata
        )

        if val_score > best_val:
            best_val = val_score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= settings["training settings"]["patience"]:
            break

    print(epoch - patience_counter)

    model.load_state_dict(best_state)

    # ------------------------------------------------------------------
    # Test
    # ------------------------------------------------------------------
    model.eval()

    with torch.no_grad():
        
        logits = model_forward(
            model,
            X_test_raw.to(device),
            X_test_extracted.to(device),
        )

        prob = torch.softmax(logits, dim=1).cpu()
        
        _, preds = torch.max(prob, dim=1)

        preds = preds.cpu()
        y_test = y_test.cpu()
        
        acc = compute_accuracy(y_test,preds)

    return acc

In [14]:
def save_results(study, trial):
    study.trials_dataframe().to_csv(
        f"{study_name}_trials.csv",
        index=False
    )

In [11]:
study_name = "lstm_per_subject_full_data_search"

In [ ]:
def objective(trial):
    metadata = {
        "eval_metric": EvalMetric.accuracy,
    }

    settings = {
        "training settings": {
            "learning rate": trial.suggest_float("lr", 0.00005, 0.00025, log=True),
            "weight decay": trial.suggest_float("wd", 0.00005, 0.00032, log=True),
    
            "patience": 15,
            "epoch": 150,
            
            "weight": None,
            "label smoothing": trial.suggest_float("ls", 0, 0.5),
            "batch size": trial.suggest_categorical("batch", [32, 64, 128, 256]),
            "accumulation_steps": trial.suggest_categorical("accumulation", [1, 2, 4]),
        },
        "model settings": {
            "model": ModelType.LSTM,
            "activation_fn": "gelu",
            "dropout": trial.suggest_float("dp", 0, 0.5), 
            "output_channels": 0,
            "output_c_multip": 0,
            "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128, 256, 512]),
            "num_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
        }
    }

    
    model_type = settings["model settings"]["model"]
    
    use_raw = model_type in [
        ModelType.CNN,
        ModelType.LSTM,
        ModelType.CNN_LSTM,
        ModelType.LSTM_CNN,
        ModelType.CNN_Transformer,
        ModelType.CNN_LSTM_Fusion,
    ]
    
    use_engineered = model_type in [
        ModelType.MLP,
        ModelType.CNN_Transformer,
    ]
    
    files_suffix = "_raw" if use_raw else "_extracted"

    cache_hash = "7f614e721ed6b83d1d2a0c91f5a7c0ef"
    cache_dir = f"{data_path}Final Training Data/Windowed Data/{cache_hash}"
    
    subjects = get_subjects(cache_dir, use_raw, use_engineered)
    
    accs = []

    
    for subject_idx, test_subject in enumerate(subjects):
            
        test_subject = test_subject.removesuffix(files_suffix)
        print(f"\nSubject {subject_idx}: {test_subject}")
        print("Loading data...")
            
        X_raw, X_engineered, y, relative_windows_times = load_subject(cache_dir, test_subject, use_raw, use_engineered)
    
        print("Training...")
        acc = train(
            X_raw, X_engineered, y, 
            relative_windows_times,
            subject_idx, test_subject,
            settings, metadata
        )
        accs.append(acc)
    return sum(accs) / len(accs)
        
study = optuna.create_study(
    study_name=study_name,
    storage="sqlite:///optuna.db",
    load_if_exists=True,
    direction="maximize",
)
study.optimize(objective, n_trials=100, callbacks=[save_results])

[I 2026-08-14 11:02:21,848] Using an existing study with name 'lstm_per_subject_full_data_search' instead of creating a new one.



Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
19

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
11

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
28

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
10

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
23

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
8

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
22

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
5

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
9

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
6

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
0

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
8

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee869

[I 2026-08-14 11:10:03,274] Trial 79 finished with value: 0.7531477083466254 and parameters: {'lr': 0.00011243494256427934, 'wd': 5.292411535408127e-05, 'ls': 0.041265688294039565, 'batch': 32, 'accumulation': 1, 'dp': 0.09438151231662821, 'h_size': 512, 'n_layers': 2}. Best is trial 69 with value: 0.7602693995333553.


23

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
45

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
1

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
22

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
39

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
89

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
43

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
35

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
0

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
55

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
0

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
21

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
32

Subject 12: 762be8f2-4fec-4ea5-bdbe-6

[I 2026-08-14 11:17:54,594] Trial 80 finished with value: 0.7045225425460597 and parameters: {'lr': 9.577540364191839e-05, 'wd': 7.66003679170012e-05, 'ls': 0.09839474662801007, 'batch': 32, 'accumulation': 4, 'dp': 0.062495406725663324, 'h_size': 64, 'n_layers': 2}. Best is trial 69 with value: 0.7602693995333553.


...

17

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
3

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
9

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
21

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
11

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
21

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
33

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
2

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
4

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
11

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
14

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
1

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
14

Subject 12: 762be8f2-4fec-4ea5-bdbe-6ae

[I 2026-08-14 13:29:53,336] Trial 94 finished with value: 0.7537071135733151 and parameters: {'lr': 7.949859438675407e-05, 'wd': 9.074561542229274e-05, 'ls': 0.1247007588782552, 'batch': 32, 'accumulation': 1, 'dp': 0.08632837055329097, 'h_size': 512, 'n_layers': 2}. Best is trial 69 with value: 0.7602693995333553.


22

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
9

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
5

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
22

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
17

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
14

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
26

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
13

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
9

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
5

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
15

Subject 10: 71748750-99ec-4f47-a314-ebb573f9769d
Loading data...
Training...
2

Subject 11: 717ba429-b68d-4649-872a-9c0f50548f08
Loading data...
Training...
8

Subject 12: 762be8f2-4fec-4ea5-bdbe-6aee

[I 2026-08-14 13:41:09,990] Trial 95 finished with value: 0.7513979950814809 and parameters: {'lr': 9.632241816813107e-05, 'wd': 8.931185363849222e-05, 'ls': 0.11296007333119143, 'batch': 32, 'accumulation': 1, 'dp': 0.0685268906542201, 'h_size': 512, 'n_layers': 2}. Best is trial 69 with value: 0.7602693995333553.


38

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
4

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...


In [11]:
CSV_PATH = f"Final/Subject_non-specific/mlp_final.csv"
SCORE_COLUMN = "value"
 
df = pl.read_csv(CSV_PATH)

# Keep only completed trials (if the column exists)
if "state" in df.columns:
    df = df.filter(pl.col("state") == "COMPLETE")

best_values = df[ df["value"].arg_max() ]
best_score = best_values.select(pl.col("value")).item()
tolerance = 0.000

while True:
    best_df = df.filter(pl.col("value") >= best_score - tolerance)

    if best_df.height >= 10:
        break

    tolerance += 0.001

best_df = df.filter(
    pl.col(SCORE_COLUMN) >= best_score - tolerance
)

param_columns = [c for c in best_values.columns if c.startswith("params_")]

print(f"Best score: {best_score:.6f}")
for param in param_columns:
    val = (
        best_values.select(
            pl.col(param),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"{val}")

print(f"Trials kept: {best_df.height} / {df.height}")
print(f"Min score: {best_score - tolerance}")
param_columns = [c for c in df.columns if c.startswith("params_")]

print("\nParameter ranges:")
for param in param_columns:
    min_val, max_val = (
        best_df.select(
            pl.col(param).min().alias("min"),
            pl.col(param).max().alias("max"),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"min={min_val}    max={max_val}")

Best score: 0.447855
accumulation         (1,)
batch                (32,)
dp                   (0.3696142771836686,)
lr                   (0.00011635719281350313,)
ls                   (0.3457883676751952,)
out_ch               (256,)
out_ch_mult          (1,)
wd                   (0.00012300424303877925,)
Trials kept: 11 / 38
Min score: 0.44585529408428315

Parameter ranges:
accumulation         min=1    max=2
batch                min=32    max=64
dp                   min=0.10427316576740453    max=0.4414475861135962
lr                   min=0.00010702025714882838    max=0.0001380606041817997
ls                   min=0.09936339303668483    max=0.3457883676751952
out_ch               min=32    max=256
out_ch_mult          min=1    max=1
wd                   min=8.188705817437177e-05    max=0.00030742694007520395
